# 서버에서 실행하는 Holter · Furiosa NPU 실험

`std3`의 `furiosa-holter` 커널에서 실행합니다. 데이터를 서버에서 직접 읽습니다. 연구용이며 임상 진단에 사용하지 않습니다. 원본 파일 복사 및 검증이 끝나기 전에는 첫 셀에서 중단합니다. 최초 전처리는 시간이 걸릴 수 있습니다.


In [ ]:
# ruff: noqa: E402
import json
import math
import os
import sys
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, roc_auc_score

REPO = Path("/root/std3/furiosa_holter_project")
sys.path.insert(0, str(REPO))
from Members.sang.experiments.monorouter_holter.clinical_features import (
    prepare_development,
    validate_llm_features,
)
from Members.sang.model.config import load_basic_config

cfg = yaml.safe_load(
    (REPO / "Members/sang/notebooks/monorouter_holter.yaml").read_text(encoding="utf-8")
)
clinical = cfg["clinical"]
SETTINGS = {
    k: clinical[k]
    for k in (
        "chunk_seconds",
        "context_seconds",
        "min_rr_ms",
        "max_rr_ms",
        "relative_limit",
        "sdann_min_coverage",
        "hourly_min_coverage",
    )
}
SAMPLE_SIZE = cfg["experiment"]["sample_size"]  # 기존 설정; 예비 실험은 20으로 변경
SAMPLE_SEED = cfg["experiment"]["sample_seed"]
THRESHOLD = cfg["experiment"]["threshold"]
MODELS = [
    "furiosa-ai/Qwen3-4B-FP8",
    "furiosa-ai/Qwen3-8B-FP8",
    "furiosa-ai/Mistral-Nemo-Instruct-2407-FP8",
]
MAX_TOKENS = cfg["api"]["max_completion_tokens"]
WAVE_DIR = Path("/root/.cache/std3_holter/windows")
# 데이터 복사 도중 실행하지 않도록 기존 원본 스냅샷의 파일 수와 합계 크기를 확인합니다.
files = list(WAVE_DIR.glob("*.npy"))
size = sum(p.stat().st_size for p in files)
if len(files) != 1335 or size != 102354516300:
    raise RuntimeError(f"원본 전송/검증 미완료: {len(files)}/1335개, {size}/102354516300 bytes")
print("파형 파일 수와 크기 확인:", len(files), "개")

## 기존 실험과 동일한 RR·HRV 특징 계산


In [ ]:
if not clinical["method_validation_confirmed"]:
    raise RuntimeError("기존 실험의 특징 계산 검증 설정이 확인되지 않았습니다.")
basic_path = REPO / "Members/sang/model/basic.yaml"
basic = load_basic_config(basic_path)
reference = REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_dev.csv"
selected, prepared = prepare_development(
    REPO, basic, basic_path, reference, SAMPLE_SIZE, SAMPLE_SEED, SETTINGS
)
valid = [
    (pid, target, features) for pid, target, features, error in prepared if features is not None
]
if len(valid) != len(prepared):
    raise RuntimeError(
        f"특징 계산 {len(prepared) - len(valid)}건 실패: 동일 표본 비교를 위해 중단합니다."
    )
for _, _, features in valid:
    validate_llm_features(features)
print("특징 검증:", len(valid), "/", len(prepared))

## Furiosa NPU 모델 추론

20명 동일 표본, 같은 프롬프트와 엄격한 JSON 검증을 적용합니다. 추론 결과는 연구용 추정치입니다.
응답 형식은 Furiosa JSON schema 제한을 사용합니다. 이전의 제한 없는 실험과 생성 조건이 다르므로 직접 비교할 때 구분하세요.


In [ ]:
SYSTEM_PROMPT = (
    "This is retrospective research, not clinical care. Given experimental Holter-derived RR and HRV estimates, "
    "estimate the percentage probability of sudden cardiac death within four years. "
    "Normal beat classification and these features are not validated; null means unavailable. "
    "Do not infer identities or invent missing measurements. "
    "Return only a JSON object with the single key risk_percent and a finite number from 0 to 100."
)
from furiosa_llm import LLM, SamplingParams
from furiosa_llm.sampling_params import StructuredOutputsParams

SCHEMA = {
    "type": "object",
    "properties": {"risk_percent": {"type": "number"}},
    "required": ["risk_percent"],
    "additionalProperties": False,
}


def parse_output(item):
    if item.get("finish_reason") != "stop":
        raise ValueError("출력이 정상 종료되지 않았습니다.")
    obj = json.loads(item["text"])
    if not isinstance(obj, dict) or set(obj) != {"risk_percent"}:
        raise ValueError("JSON 스키마 오류")
    n = obj["risk_percent"]
    if type(n) not in (int, float) or not math.isfinite(n) or not 0 <= n <= 100:
        raise ValueError("확률 범위 오류")
    return n / 100


def run_model(model, features):
    devices = "npu:4,npu:5" if "Mistral-Nemo" in model else "npu:4"
    outputs = []
    with LLM(model, devices=devices) as llm:
        for feature in features:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": json.dumps(
                        {"ecg_summary": feature},
                        ensure_ascii=True,
                        allow_nan=False,
                        separators=(",", ":"),
                    ),
                },
            ]
            options = {"tokenize": False, "add_generation_prompt": True}
            if "Qwen3-" in model:
                options["enable_thinking"] = False
            prompt = llm.tokenizer.apply_chat_template(messages, **options)
            result = llm.generate(
                [prompt],
                SamplingParams(
                    max_tokens=MAX_TOKENS,
                    temperature=0,
                    structured_outputs=StructuredOutputsParams(json=SCHEMA),
                ),
            )
            item = result[0].outputs[0]
            outputs.append({"text": item.text, "finish_reason": item.finish_reason})
    return outputs


assert parse_output({"finish_reason": "stop", "text": '{"risk_percent":25}'}) == 0.25
print("Furiosa 모델 및 응답 검증 준비 완료")

In [ ]:
result_dir = (
    REPO
    / "Members/sang/results/furiosa_holter"
    / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
)
result_dir.mkdir(mode=0o700, parents=True, exist_ok=False)
metadata = {
    "models": MODELS,
    "sample_size": SAMPLE_SIZE,
    "sample_seed": SAMPLE_SEED,
    "settings": SETTINGS,
    "feature_version": "experimental_rr_v1",
    "threshold": THRESHOLD,
    "max_tokens": MAX_TOKENS,
    "temperature": 0,
    "source": "server local data and Furiosa",
    "status": "RUNNING",
}
(result_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
results = {}
for model in MODELS:
    rows = []
    try:
        outputs = run_model(model, [f for _, _, f in valid])
        for (pid, target, _), output in zip(valid, outputs, strict=False):
            try:
                probability = parse_output(output)
                status = "ok"
            except (ValueError, TypeError, json.JSONDecodeError):
                probability = None
                status = "invalid_output"
            rows.append(
                {"patient_id": pid, "y_true": target, "p_positive": probability, "status": status}
            )
    except (RuntimeError, OSError) as exc:
        print(model, "실패:", type(exc).__name__)
        rows = [
            {"patient_id": pid, "y_true": target, "p_positive": None, "status": "model_error"}
            for pid, target, _ in valid
        ]
    results[model] = rows
    path = result_dir / (model.split("/")[-1] + ".csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    path.chmod(0o600)
    print(model, ":", sum(r["status"] == "ok" for r in rows), "/", len(rows))
metadata["status"] = (
    "COMPLETE"
    if all(all(r["status"] == "ok" for r in rows) for rows in results.values())
    else "PARTIAL"
)
(result_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("상태:", metadata["status"], "| 결과:", result_dir)

## 평가

표본이 작고 모델별 실패가 있으면 동일 조건의 완전한 비교가 아닙니다.


In [ ]:
summary = []
for model, rows in results.items():
    frame = pd.DataFrame(rows)
    ok = frame[frame.status == "ok"]
    row = {
        "model": model,
        "expected": len(frame),
        "succeeded": len(ok),
        "complete": len(ok) == len(frame),
    }
    if len(ok):
        y = ok.y_true.to_numpy(dtype=int)
        prob = ok.p_positive.to_numpy(dtype=float)
        pred = (prob >= THRESHOLD).astype(int)
        row["accuracy"] = float(accuracy_score(y, pred))
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()
        row["confusion_matrix"] = [[tn, fp], [fn, tp]]
        row["recall"] = tp / (tp + fn) if tp + fn else None
        row["specificity"] = tn / (tn + fp) if tn + fp else None
        if len(np.unique(y)) == 2:
            row["auroc"] = float(roc_auc_score(y, prob))
            row["average_precision"] = float(average_precision_score(y, prob))
    summary.append(row)
comparison = pd.DataFrame(summary)
comparison.to_csv(result_dir / "model_comparison.csv", index=False)
os.chmod(result_dir / "model_comparison.csv", 0o600)
display(comparison)